---
title: "Лекція 2. Узагальнення, невизначеність і межі статистичного висновку в машинному навчанні"
description: "Машинне навчання та методи вилучення знань | PhD F3 «Комп'ютерні науки» | КрНУ ім. М. Остроградського"
author: "&copy; [Valeriy Sydorenko](https://www.linkedin.com/in/valeriy-sydorenko-6782279a/), 2026"
date: today
jupyter: python3
bibliography: mlkdd-references.bib

format:
  html:
    code-fold: show
    code-summary: "Показати код"
    code-overflow: wrap
    toc: true
    toc-depth: 4
    toc-location: right
    number-sections: false
    embed-resources: true
    smooth-scroll: true
---

## Мета лекції

Сформувати системне розуміння *узагальнення (generalization)* як центральної проблеми машинного навчання, джерел невизначеності ML-результатів та меж статистичного висновку, а також навчити розрізняти високу прогнозну якість моделі й достатність доказів для наукового твердження.

Після лекції здобувач має вміти:

- пояснювати різницю між емпіричним ризиком і ризиком узагальнення;
- розрізняти невизначеність даних, моделі, параметрів і прогнозу;
- пояснювати роль вибірки та припущення про її походження;
- розпізнавати *зміщення розподілу (distribution shift)* і дані поза межами розподілу (out-of-distribution, OOD);
- визначати область застосовності моделі;
- критично інтерпретувати тестові метрики та їхню невизначеність;
- встановлювати межі наукового висновку, який підтримує ML-експеримент.

> Центральне питання лекції: **наскільки далеко за межі конкретної навчальної та тестової вибірки ми маємо право переносити отриманий ML-результат?**

## 1. Від «модель добре працює» до питання «де саме вона працює?»

У Лекції 1 ми зафіксували:

$$
\text{висока прогнозна якість}
\not\Rightarrow
\text{достатній науковий доказ}.
$$

Тепер уточнимо, чому.

Нехай спостереження $(X,Y)$ породжуються деяким невідомим розподілом $P$. Для моделі $f$ і функції втрат $L$ ідеально нас цікавить *очікуваний ризик*

$$
R(f)
=
\mathbb{E}_{(X,Y)\sim P}
\left[
L(Y,f(X))
\right].
$$

Але розподіл $P$ нам безпосередньо недоступний. Ми маємо лише скінченну вибірку

$$
D=\{(x_i,y_i)\}_{i=1}^{n}
$$

і можемо обчислити *емпіричний ризик*

$$
\hat R_D(f)
=
\frac{1}{n}
\sum_{i=1}^{n}
L(y_i,f(x_i)).
$$

Центральна проблема статистичного навчання полягає не в тому, щоб зробити $\hat R_D(f)$ малим, а в тому, щоб на підставі скінченної вибірки отримати підстави очікувати малий $R(f)$ [@vapnik1998statistical; @hastie2009elements].

## 2. Узагальнення як перехід від вибірки до популяції

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart LR
    P("Цільова популяція<br/>P(X,Y)") --> S("Вибірка<br/>D")
    S --> T("Навчальні<br/>дані")
    S --> E("Тестові<br/>дані")
    T --> M1("Навчена<br/>модель")
    M1 --> Q("Оцінка якості<br/>на test")
    Q --> I("Висновок про<br/>майбутні дані")
    I -. "потребує припущень" .-> P
```

Остання стрілка є методологічно найважливішою.

Тестова вибірка не «представляє реальність» автоматично. Перехід від тестового результату до твердження про цільову популяцію спирається на припущення про механізм формування даних.

## 3. Які припущення приховані за звичайним train/test split?

У найпростішій теорії часто припускають, що спостереження є *незалежними та однаково розподіленими (independent and identically distributed, i.i.d.)*:

$$
(X_i,Y_i)\overset{i.i.d.}{\sim}P(X,Y).
$$

Це означає дві різні вимоги:

1. спостереження не повинні створювати неконтрольовану залежність;
2. навчальні, тестові та майбутні дані походять з одного релевантного розподілу.

У реальних дослідженнях обидва припущення можуть порушуватися.

Приклади:

- декілька записів належать одному пацієнту;
- повідомлення належать одному користувачу;
- сусідні часові точки автокорельовані;
- зображення одного об'єкта потрапили і в train, і в test;
- модель навчається на даних одного року, а використовується через два роки;
- train сформовано в одній установі, а inference відбувається в іншій.

Тому випадковий `train_test_split()` -- це *дизайнерське рішення*, а не нейтральна технічна операція.

## 4. Помилка навчання і помилка узагальнення

Введемо *розрив узагальнення (generalization gap)*:

$$
G(f)
=
R(f)-\hat R_{\text{train}}(f).
$$

На практиці $R(f)$ невідомий, тому його наближено оцінюють на незалежних даних:

$$
\widehat{G}
=
\hat R_{\text{test}}(f)
-
\hat R_{\text{train}}(f).
$$

Великий розрив часто сигналізує про перенавчання (overfitting), але малий розрив **не гарантує** зовнішньої валідності.

Наприклад, train і test можуть бути однаково зміщеними відносно цільової популяції. Тоді модель добре узагальнюється *всередині помилково визначеного розподілу*, але не на ту область, про яку дослідник хоче зробити висновок.

## 5. Bias--variance як інтуїція, а не повна картина

Для регресії з квадратичною втратою очікувану похибку часто концептуально розкладають на

$$
\mathbb{E}\left[(Y-\hat f(X))^2\right]
=
\sigma^2
+
\operatorname{Bias}^2[\hat f(X)]
+
\operatorname{Var}[\hat f(X)].
$$

Тут:

- $\sigma^2$ -- невідворотний шум;
- bias -- систематична похибка через обмеження моделі;
- variance -- чутливість оціненої моделі до конкретної навчальної вибірки.

Ця схема корисна для розуміння overfitting, але в науковому дослідженні існують додаткові джерела невизначеності: зміщення вибірки, похибка вимірювання, невизначеність представлення, вибір метрики, distribution shift і дослідницькі рішення.

## 6. Карта невизначеності ML-дослідження

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart TD
    U("Невизначеність<br/>ML-дослідження") --> D("Дані")
    U --> M2("Модель")
    U --> P("Параметри")
    U --> PR("Прогноз")
    U --> DS("Розподіл")
    D --> D1("вибірка")
    D --> D2("вимірювання")
    M2 --> M3("клас моделей")
    M2 --> M4("гіперпараметри")
    P --> P1("оцінювання")
    PR --> P2("індивідуальний прогноз")
    DS --> DS1("distribution shift")
    DS --> DS2("OOD")
```

## 7. Алеаторна та епістемічна невизначеність

У ML часто розрізняють два концептуальні типи невизначеності.

### Алеаторна невизначеність (aleatoric uncertainty)

Пов'язана з варіативністю самого процесу або шумом спостереження. Навіть за ідеального знання моделі результат може залишатися випадковим.

У простій регресії:

$$
Y=f(X)+\varepsilon,
\qquad
\varepsilon\sim\mathcal{N}(0,\sigma^2).
$$

### Епістемічна невизначеність (epistemic uncertainty)

Пов'язана з неповнотою нашого знання: обмеженою вибіркою, невідомою функцією, вибором класу моделей, параметрами.

Умовно:

$$
\text{невизначеність прогнозу}
=
\text{невизначеність процесу}
+
\text{невизначеність знання}.
$$

Цей поділ корисний, але не слід сприймати його як універсальне й однозначне розкладання для будь-якої ML-системи.

## 8. Невизначеність метрики

Навіть якщо тестовий набір незалежний, отримана метрика є *оцінкою*, а не константою.

Нехай accuracy на тесті:

$$
\hat p=\frac{k}{n}.
$$

За ідеалізованої біноміальної постановки її стандартна похибка приблизно дорівнює

$$
SE(\hat p)
=
\sqrt{
\frac{\hat p(1-\hat p)}{n}
}.
$$

Отже, два значення accuracy 0.87 і 0.89 не обов'язково свідчать про реальну різницю між моделями. Потрібно враховувати розмір тестової вибірки та процедуру порівняння.

## 9. Приклад: одна accuracy -- різна впевненість

Розглянемо дві тестові вибірки.

### Випадок A

$$
n=100,\qquad \hat p=0.90.
$$

Тоді

$$
SE
=
\sqrt{
\frac{0.9\cdot0.1}{100}
}
=
0.03.
$$

Грубий 95% інтервал:

$$
0.90\pm1.96\cdot0.03
\approx
[0.841;\ 0.959].
$$

### Випадок B

$$
n=10\,000,\qquad \hat p=0.90.
$$

$$
SE
=
\sqrt{
\frac{0.9\cdot0.1}{10000}
}
=
0.003.
$$

Отже,

$$
0.90\pm1.96\cdot0.003
\approx
[0.894;\ 0.906].
$$

**Та сама точкова метрика 0.90 має принципово різну доказову вагу.**

## 10. Перевіримо це обчислювально

In [1]:
import numpy as np
import pandas as pd

def normal_accuracy_interval(p_hat, n, z=1.96):
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    return pd.Series({
        "accuracy": p_hat,
        "n_test": n,
        "SE": se,
        "lower_95": p_hat - z * se,
        "upper_95": p_hat + z * se
    })

pd.DataFrame([
    normal_accuracy_interval(0.90, 100),
    normal_accuracy_interval(0.90, 10_000)
]).round(4)

,accuracy,n_test,SE,lower_95,upper_95
0,0.9,100.0,0.030,0.8412,0.9588
1,0.9,10000.0,0.003,0.8941,0.9059


::: {.callout-warning}
## Методологічне застереження

Наведений нормальний інтервал -- навчальне наближення. Для малих вибірок або значень частки поблизу 0 чи 1 доцільніші, наприклад, Wilson interval або інші коректні процедури.

Головна теза тут інша: **метрика без характеристики її невизначеності є неповним результатом**.
:::

## 11. Bootstrap як універсальніша інтуїція

Для складних метрик аналітична стандартна похибка може бути незручною або невідомою. Тоді часто використовують *bootstrap*:

1. з тестової вибірки багато разів формують вибірки з поверненням;
2. для кожної обчислюють метрику;
3. аналізують емпіричний розподіл метрики.

Якщо маємо значення

$$
m^{*(1)},m^{*(2)},\ldots,m^{*(B)},
$$

то вони дають уявлення про варіативність оцінки метрики за повторного вибіркового процесу.

Bootstrap не виправляє поганий дизайн вибірки. Він оцінює невизначеність *за умов, закладених у процедуру ресемплінгу*.

In [2]:
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)

# Умовний набір істинних класів і прогнозів.
n = 250
y_true = rng.binomial(1, 0.4, size=n)

# Створимо прогнози з приблизно 85% правильних відповідей.
flip = rng.binomial(1, 0.15, size=n)
y_pred = np.where(flip == 1, 1 - y_true, y_true)

observed_accuracy = accuracy_score(y_true, y_pred)

B = 2000
bootstrap_scores = []

for _ in range(B):
    idx = rng.integers(0, n, size=n)
    bootstrap_scores.append(
        accuracy_score(y_true[idx], y_pred[idx])
    )

ci = np.quantile(bootstrap_scores, [0.025, 0.975])

pd.Series({
    "Observed accuracy": observed_accuracy,
    "Bootstrap lower 95%": ci[0],
    "Bootstrap upper 95%": ci[1]
}).round(3)

Observed accuracy      0.864
Bootstrap lower 95%    0.820
Bootstrap upper 95%    0.904
dtype: float64

## 12. Від невизначеності вибірки до невизначеності моделі

Уявімо, що ми повторюємо весь процес:

$$
D^{(1)}\rightarrow \hat f^{(1)},
\quad
D^{(2)}\rightarrow \hat f^{(2)},
\quad \ldots
$$

Якщо невелика зміна навчальної вибірки суттєво змінює модель або висновок, то результат нестабільний.

Для PhD-дослідження питання має звучати не лише:

> «Яку метрику отримала модель?»

а й:

> «Наскільки залежить цей результат від конкретної реалізації вибірки, random seed, split, представлення даних та набору гіперпараметрів?»

Це підводить нас до робастності та дизайну ML-експерименту, які будуть центральними у Лекції 3.

## 13. Distribution shift: коли майбутнє не схоже на train

Нехай під час навчання дані походять із

$$
P_{\text{train}}(X,Y),
$$

а під час застосування --

$$
P_{\text{target}}(X,Y).
$$

У класичній i.i.d.-постановці ми фактично сподіваємося, що

$$
P_{\text{train}}(X,Y)
\approx
P_{\text{target}}(X,Y).
$$

Якщо це не так, маємо *зміщення розподілу (distribution shift)*.

Загально:

$$
P_{\text{train}}(X,Y)
\neq
P_{\text{target}}(X,Y).
$$

Причиною можуть бути час, географія, інша популяція, зміна сенсорів, платформи, поведінки користувачів або самої системи, яку ми досліджуємо.

## 14. Основні форми зміщення розподілу

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart TD
    S("Distribution<br/>shift") --> C("Covariate shift")
    S --> L("Label shift")
    S --> K("Concept shift")
    C --> C1("P(X) змінюється")
    C --> C2("P(Y|X) умовно стабільне")
    L --> L1("P(Y) змінюється")
    L --> L2("P(X|Y) умовно стабільне")
    K --> K1("P(Y|X) змінюється")
```

Це концептуальна класифікація. У реальних даних механізми можуть накладатися.

## 15. Covariate shift

За *covariate shift* змінюється розподіл ознак:

$$
P_{\text{train}}(X)
\neq
P_{\text{target}}(X),
$$

але умовний механізм в ідеалізованій постановці вважається стабільним:

$$
P_{\text{train}}(Y\mid X)
=
P_{\text{target}}(Y\mid X).
$$

Приклад: модель навчили на молодшій групі користувачів, а застосовують до популяції з іншим віковим складом, хоча зв'язок між ознаками та результатом умовно не змінився.

## 16. Label shift

За *label shift* змінюється частота класів:

$$
P_{\text{train}}(Y)
\neq
P_{\text{target}}(Y),
$$

а $P(X\mid Y)$ умовно вважається стабільним.

Наприклад, частка позитивного класу в дослідницькому датасеті була штучно збалансована, тоді як у реальній популяції подія рідкісна.

У такій ситуації discrimination-метрики можуть виглядати прийнятно, але ймовірнісна інтерпретація та практичні predictive values змінюються.

## 17. Concept shift / concept drift

Найскладніша ситуація виникає, коли змінюється сам зв'язок між $X$ і $Y$:

$$
P_{\text{train}}(Y\mid X)
\neq
P_{\text{target}}(Y\mid X).
$$

Для часових систем часто використовують термін *дрейф концепту (concept drift)*.

Наприклад, поведінкові патерни, які дозволяли виявляти автоматизовані акаунти у 2024 році, можуть змінитися після адаптації ботів до детекторів у 2026 році.

Тут історична тестова вибірка може бути абсолютно коректною щодо минулого й водночас недостатньою щодо майбутнього.

## 18. Простий експеримент із distribution shift

Побудуємо модель на одному розподілі й перевіримо її:

1. на i.i.d.-тесті;
2. на зміщеному розподілі.

Модель не змінюється. Змінюється лише середовище, у якому ми її оцінюємо.

In [3]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

# Навчальний розподіл.
X, y = make_classification(
    n_samples=3000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    class_sep=1.2,
    random_state=42
)

X_train, X_iid, y_train, y_iid = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Створюємо зміщений target-набір:
# частину ознак систематично зсуваємо.
X_shift = X_iid.copy()
X_shift[:, :3] += 1.5
y_shift = y_iid.copy()

def evaluate(model, X, y):
    prob = model.predict_proba(X)[:, 1]
    pred = model.predict(X)
    return {
        "ROC AUC": roc_auc_score(y, prob),
        "Accuracy": accuracy_score(y, pred)
    }

pd.DataFrame({
    "IID test": evaluate(model, X_iid, y_iid),
    "Shifted test": evaluate(model, X_shift, y_shift)
}).round(3)

,IID test,Shifted test
ROC AUC,0.832,0.832
Accuracy,0.757,0.759


### Що демонструє приклад?

Ми не «зіпсували» модель після навчання. Ми змінили умови її застосування.

Тому твердження

> «модель має ROC AUC = ...»

є неповним.

Коректніше:

> «за визначеного способу формування тестової вибірки, що репрезентує такий-то розподіл, отримано ROC AUC = ...».

Метрика є властивістю не лише моделі, а **пари «модель -- розподіл даних»**.

## 19. Дані поза межами розподілу (OOD)

*Дані поза межами розподілу (out-of-distribution, OOD)* -- це спостереження, що суттєво відрізняються від області, представленої під час навчання.

Формально універсальної межі «in-distribution / OOD» немає. Вона залежить від:

- представлення даних;
- обраної метрики або score;
- класу моделей;
- цільового застосування.

Це особливо важливо для високовимірних даних, зображень, тексту та learned representations.

Модель може видавати числовий прогноз для OOD-об'єкта. Сам факт наявності прогнозу **не означає**, що прогноз має достатню епістемічну підставу.

## 20. Область застосовності моделі

Доцільно явно задавати *область застосовності (applicability domain)*:

$$
\mathcal{A}
\subseteq
\mathcal{X},
$$

у межах якої існують емпіричні підстави довіряти моделі.

Для об'єкта $x^*$ потрібно питати не лише

$$
\hat y=f(x^*),
$$

а й

$$
x^*\in\mathcal{A}\ ?
$$

У прикладних системах це може приводити до механізму *відмови від прогнозу (abstention)*:

$$
g(x)=
\begin{cases}
f(x), & x\in\mathcal{A},\\
\text{abstain}, & x\notin\mathcal{A}.
\end{cases}
$$

Для наукового дослідження аналогічна логіка означає: **не кожне спостереження підтримує однаково сильний висновок**.

## 21. Внутрішня, зовнішня і статистична валідність

Корисно розділяти кілька питань.

### Внутрішня валідність

Чи отриманий результат справді є наслідком коректного дизайну й аналізу в межах дослідження, а не leakage, confounding, помилки вимірювання або іншого артефакту?

### Статистична валідність висновку

Наскільки коректно оцінено величину ефекту/метрики та її невизначеність?

### Зовнішня валідність

На які популяції, часові періоди, середовища й умови можна переносити результат?

ML може мати високу внутрішню тестову якість і слабку зовнішню валідність.

## 22. Ланцюг статистичного висновку

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': '18px',
  'fontFamily': 'Trebuchet MS, Verdana, Arial, sans-serif',
  'primaryColor': '#E8F3FF',
  'primaryBorderColor': '#3478A8',
  'secondaryColor': '#FFF0C9',
  'tertiaryColor': '#E7F6EC',
  'lineColor': '#536878'
}}}%%
flowchart LR
    D("Спостережені<br/>дані") --> E("Оцінена<br/>метрика")
    E --> U("Невизначеність<br/>оцінки")
    U --> V("Внутрішня<br/>валідність")
    V --> X("Зовнішня<br/>валідність")
    X --> C("Наукове<br/>твердження")
```

Кожний перехід може звузити допустиму силу твердження.

## 23. Приклад: що саме можна стверджувати?

Припустимо:

- модель навчено на даних одного університету;
- тестова вибірка сформована випадковим відкладенням 20% тих самих даних;
- ROC AUC = 0.91;
- bootstrap 95% CI = [0.88; 0.94].

### Підтримується даними

> У межах даних, сформованих за цією процедурою, модель демонструє високу здатність ранжувати позитивні та негативні випадки.

### Потребує додаткової перевірки

> Модель матиме таку саму якість в іншому університеті.

### Не випливає з результату

> Виявлені моделлю ознаки є причинами досліджуваного явища.

Отже, числова точність твердження не компенсує методологічного стрибка за межі дизайну.

## 24. «Більше даних» не завжди розв'язує проблему

Зі збільшенням $n$ може зменшуватися вибіркова невизначеність:

$$
SE \propto \frac{1}{\sqrt{n}}.
$$

Але якщо дані систематично зміщені, великий $n$ лише дає *дуже точну оцінку для неправильного цільового розподілу*.

Схематично:

$$
\text{великий }n
+
\text{systematic bias}
\neq
\text{валідний висновок}.
$$

Це принципово для Big Data: обсяг даних не замінює репрезентативність і коректний дизайн.

## 25. Prediction interval, confidence interval і uncertainty

Терміни часто змішують.

*Довірчий інтервал (confidence interval)* зазвичай характеризує невизначеність оцінювання параметра або агрегованої характеристики.

*Прогнозний інтервал (prediction interval)* характеризує діапазон можливих значень нового спостереження і зазвичай ширший, бо включає індивідуальну варіативність.

У простій регресії:

$$
Y^*
=
f(x^*)+\varepsilon^*.
$$

Навіть якщо $f(x^*)$ оцінено дуже точно, $\varepsilon^*$ не зникає.

Тому «ми впевнені у середній залежності» та «ми точно прогнозуємо конкретний об'єкт» -- різні твердження.

## 26. Калібрування: 0.9 має означати приблизно 90%

Для ймовірнісного класифікатора важлива не лише discrimination, а й *калібрування (calibration)*.

Ідеально:

$$
P(Y=1\mid \hat p(X)=p)\approx p.
$$

Якщо серед об'єктів, яким модель присвоїла $\hat p\approx0.8$, позитивними є приблизно 80%, модель добре калібрована в цій області.

ROC AUC може бути високою навіть за поганого калібрування. Тому вибір метрики залежить від того, яке саме наукове або прикладне твердження ми хочемо робити.

## 27. Чому test set не повинен перетворюватися на train set

Якщо дослідник багато разів:

1. оцінює модель на test;
2. змінює preprocessing, features або hyperparameters;
3. знову дивиться на той самий test;

то test поступово впливає на процес прийняття рішень.

У методологічному сенсі відбувається *адаптація до тесту*. Формально дані можуть не входити у `.fit()`, але інформація про них входить у дослідницький процес.

Це один із проявів ширшої проблеми *researcher degrees of freedom*, яку детально розглядатимемо в Лекції 3.

## 28. Межі статистичного висновку

Статистичний/ML-висновок завжди має умовну форму:

> **за таких даних, такого механізму вибірки, таких припущень, такого протоколу оцінювання та визначеної області застосовності** отримано такий результат.

Втрата умовної частини перетворює локальний емпіричний результат на надмірне універсальне твердження.

Тому замість:

> «Модель довела, що ...»

доцільніше формулювати:

> «За визначеного дизайну отримані результати підтримують твердження, що ... у межах ...».

## 29. Чекліст перед науковим твердженням

Перед перенесенням ML-результату на науковий висновок поставте питання:

| Блок | Питання |
|---|---|
| Вибірка | Як сформовано дані? |
| Незалежність | Чи справді одиниці аналізу незалежні? |
| Розподіл | Чи відповідає test цільовому середовищу? |
| Метрика | Що саме вимірює обрана метрика? |
| Невизначеність | Який діапазон правдоподібних значень метрики? |
| Стабільність | Чи залежить результат від split/seed? |
| Shift | Які зміни розподілу очікуються? |
| OOD | Як визначено область застосовності? |
| Валідність | На яку популяцію можна переносити результат? |
| Твердження | Чи не сильніше воно за наявні докази? |

## 30. Зв'язок із лабораторною роботою 2

ЛР2 має назву:

> **«Побудова базової моделі та проєктування ML-експерименту»**.

У ЛР1 було створено **ML/KDD Research Specification v0.1**.

Тепер потрібно перейти від специфікації до експериментального дизайну:

$$
RQ
\rightarrow
дані
\rightarrow
baseline
\rightarrow
split/validation
\rightarrow
метрики
\rightarrow
uncertainty
\rightarrow
протокол.
$$

Ключова вимога ЛР2: експериментальний протокол має бути визначений *до того*, як результати складнішої моделі почнуть впливати на дослідницькі рішення.

## 31. Підсумки

1. **Узагальнення -- це перехід від скінченної вибірки до твердження про цільовий розподіл.**
2. Train/test split має сенс лише разом із припущеннями про походження даних.
3. Тестова метрика є випадковою оцінкою і потребує характеристики невизначеності.
4. Невизначеність ML-дослідження не зводиться до одного confidence interval.
5. Distribution shift порушує просту логіку «train і майбутнє мають однаковий розподіл».
6. OOD-прогноз може бути чисельно визначеним, але епістемічно ненадійним.
7. Область застосовності моделі потрібно формулювати явно.
8. Великий обсяг даних зменшує випадкову похибку, але не усуває систематичне зміщення.
9. Висока прогнозна якість не є достатньою умовою зовнішньої валідності.
10. **Сила наукового твердження не повинна перевищувати силу дизайну та доказів.**

## Питання для самоконтролю

1. Чим емпіричний ризик відрізняється від очікуваного ризику?
2. Що означає узагальнення у статистичному навчанні?
3. Які припущення приховані за звичайним випадковим train/test split?
4. Що таке generalization gap?
5. Чому малий generalization gap не гарантує зовнішньої валідності?
6. Чим алеаторна невизначеність відрізняється від епістемічної?
7. Чому точкова метрика без характеристики невизначеності є неповним результатом?
8. Яку роль може відігравати bootstrap?
9. Що таке distribution shift?
10. Чим covariate shift, label shift і concept shift відрізняються один від одного?
11. Що означає OOD?
12. Що таке область застосовності моделі?
13. Чому модель інколи повинна мати можливість відмовитися від прогнозу?
14. Чим внутрішня валідність відрізняється від зовнішньої?
15. Чому «більше даних» не усуває systematic bias?
16. Чим confidence interval відрізняється від prediction interval?
17. Що таке calibration і чому ROC AUC її не замінює?
18. Як багаторазове використання test set може зруйнувати його роль?
19. Які умови потрібно вказувати разом із науковим ML-висновком?
20. Як поняття цієї лекції мають вплинути на дизайн ЛР2?

## Література

::: {#refs}
:::